<a href="https://colab.research.google.com/github/narakrm/northstar_databases_analytics/blob/main/Section8_Query_Optimisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Section 8 — Query Optimisation
**NorthStar Urban Mobility and Logistics — Databases and Analytics Assignment**

This notebook implements indexing strategies for the three MongoDB collections and evaluates their impact using `explain('executionStats')`.

**Indexes implemented:**

| Index | Collection | Field(s) | Type |
|-------|-----------|----------|------|
| I1 | customer_cases | customer_id | Single (unique) |
| I2 | customer_cases | home_zone | Single |
| I3 | fleet_health | maintenance_status | Single |
| I4 | fleet_health | assigned_zone + maintenance_status | Compound |
| I5 | operational_events | customer_id | Single |

---
> **Prerequisite:** Run the Section 7 notebook first to populate all three collections.  
> **Atlas URI:** Store your connection string in Colab Secrets as `MONGO_URI`.

## Cell 1 — Connect and verify collections exist

In [ ]:
!pip install pymongo --quiet

from pymongo import MongoClient, ASCENDING
from google.colab import userdata
import pprint

MONGO_URI = userdata.get('MONGO_URI')
client = MongoClient(MONGO_URI)
client.admin.command('ping')
print('Connected to MongoDB Atlas')

db                 = client['northstar_db']
customer_cases     = db['customer_cases']
operational_events = db['operational_events']
fleet_health       = db['fleet_health']

# Verify collections
for name in ['customer_cases', 'operational_events', 'fleet_health']:
    count = db[name].count_documents({})
    print(f'  {name}: {count} documents')

---
## Cell 2 — Helper: explain() summary function

This helper prints the key execution statistics from `explain('executionStats')` in a clean format.

In [ ]:
def explain_summary(collection, query, label, projection=None):
    """Run explain() and print key execution statistics."""
    cursor = collection.find(query, projection) if projection else collection.find(query)
    result = cursor.explain('executionStats')

    plan  = result['queryPlanner']['winningPlan']
    stats = result['executionStats']

    # Extract stage — IXSCAN is nested under FETCH
    stage = plan.get('stage', 'UNKNOWN')
    if stage == 'FETCH' and 'inputStage' in plan:
        stage = plan['inputStage'].get('stage', stage)
    elif stage == 'PROJECTION_SIMPLE' and 'inputStage' in plan:
        inner = plan['inputStage']
        if inner.get('stage') == 'FETCH' and 'inputStage' in inner:
            stage = inner['inputStage'].get('stage', stage)
        else:
            stage = inner.get('stage', stage)

    docs_examined = stats['totalDocsExamined']
    docs_returned = stats['totalDocsReturned']
    time_ms       = stats['executionTimeMillis']
    selectivity   = round(docs_returned / max(docs_examined, 1), 3)

    print(f'--- {label} ---')
    print(f'  Stage:           {stage}')
    print(f'  docsExamined:    {docs_examined}')
    print(f'  docsReturned:    {docs_returned}')
    print(f'  executionTimeMs: {time_ms}')
    print(f'  selectivity:     {selectivity}')
    print()

    return {'stage': stage, 'docsExamined': docs_examined,
            'docsReturned': docs_returned, 'timeMs': time_ms}

print('Helper function defined.')

---
## Cell 3 — Baseline: unindexed performance (COLLSCAN)

Before creating any indexes, all queries result in a COLLSCAN — MongoDB examines every document in the collection.
- `customer_cases`: 650 documents
- `fleet_health`: 120 documents  
- `operational_events`: 640 documents

In [ ]:
# Drop any existing indexes (keep _id index only) for clean baseline
customer_cases.drop_indexes()
fleet_health.drop_indexes()
operational_events.drop_indexes()
print('All custom indexes dropped — baseline (COLLSCAN) state\n')

print('BASELINE — no indexes\n')

baseline = {}
baseline['I1'] = explain_summary(
    customer_cases,
    {'customer_id': 'C0182'},
    'Baseline I1: find customer by customer_id'
)
baseline['I2'] = explain_summary(
    customer_cases,
    {'home_zone': 'North'},
    'Baseline I2: find all North zone customers'
)
baseline['I3'] = explain_summary(
    fleet_health,
    {'maintenance_status': 'InRepair'},
    'Baseline I3: find all InRepair vehicles'
)
baseline['I4'] = explain_summary(
    fleet_health,
    {'assigned_zone': 'Airport', 'maintenance_status': 'InRepair'},
    'Baseline I4: find Airport InRepair vehicles'
)
baseline['I5'] = explain_summary(
    operational_events,
    {'customer_id': 'C0144'},
    'Baseline I5: find sessions for customer C0144'
)

---
## Cell 4 — Index I1: customer_cases — customer_id (unique single field)

In [ ]:
# Create index
customer_cases.create_index(
    [('customer_id', ASCENDING)],
    name='idx_customer_id',
    unique=True
)
print('Index I1 created: customer_cases.customer_id (unique)\n')

# Explain after
after_i1 = explain_summary(
    customer_cases,
    {'customer_id': 'C0182'},
    'After I1: find customer by customer_id'
)

improvement = round((1 - after_i1['docsExamined'] / max(baseline['I1']['docsExamined'], 1)) * 100, 1)
print(f'Improvement: {baseline["I1"]["docsExamined"]} -> {after_i1["docsExamined"]} docs examined ({improvement}% reduction)')

## Cell 5 — Index I2: customer_cases — home_zone (single field)

In [ ]:
customer_cases.create_index(
    [('home_zone', ASCENDING)],
    name='idx_home_zone'
)
print('Index I2 created: customer_cases.home_zone\n')

after_i2 = explain_summary(
    customer_cases,
    {'home_zone': 'North'},
    'After I2: find all North zone customers'
)

improvement = round((1 - after_i2['docsExamined'] / max(baseline['I2']['docsExamined'], 1)) * 100, 1)
print(f'Improvement: {baseline["I2"]["docsExamined"]} -> {after_i2["docsExamined"]} docs examined ({improvement}% reduction)')
print(f'Note: North zone has 111 customers — index narrows scan to exactly that subset')

## Cell 6 — Index I3: fleet_health — maintenance_status (single field)

In [ ]:
fleet_health.create_index(
    [('maintenance_status', ASCENDING)],
    name='idx_maintenance_status'
)
print('Index I3 created: fleet_health.maintenance_status\n')

after_i3 = explain_summary(
    fleet_health,
    {'maintenance_status': 'InRepair'},
    'After I3: find all InRepair vehicles'
)

improvement = round((1 - after_i3['docsExamined'] / max(baseline['I3']['docsExamined'], 1)) * 100, 1)
print(f'Improvement: {baseline["I3"]["docsExamined"]} -> {after_i3["docsExamined"]} docs examined ({improvement}% reduction)')
print(f'Note: 36 of 120 vehicles are InRepair — index narrows scan directly to those 36')

## Cell 7 — Index I4: fleet_health — compound (assigned_zone, maintenance_status)

In [ ]:
fleet_health.create_index(
    [('assigned_zone', ASCENDING), ('maintenance_status', ASCENDING)],
    name='idx_zone_maintenance'
)
print('Index I4 created: fleet_health (assigned_zone, maintenance_status) compound\n')

after_i4 = explain_summary(
    fleet_health,
    {'assigned_zone': 'Airport', 'maintenance_status': 'InRepair'},
    'After I4: find Airport InRepair vehicles'
)

improvement = round((1 - after_i4['docsExamined'] / max(baseline['I4']['docsExamined'], 1)) * 100, 1)
print(f'Improvement: {baseline["I4"]["docsExamined"]} -> {after_i4["docsExamined"]} docs examined ({improvement}% reduction)')
print(f'Note: Only 7 Airport InRepair vehicles — compound index navigates directly to them')
print(f'Note: assigned_zone placed first as higher-cardinality prefix for this query pattern')

## Cell 8 — Index I5: operational_events — customer_id (single field)

In [ ]:
operational_events.create_index(
    [('customer_id', ASCENDING)],
    name='idx_events_customer_id'
)
print('Index I5 created: operational_events.customer_id\n')

after_i5 = explain_summary(
    operational_events,
    {'customer_id': 'C0144'},
    'After I5: find all sessions for customer C0144'
)

improvement = round((1 - after_i5['docsExamined'] / max(baseline['I5']['docsExamined'], 1)) * 100, 1)
print(f'Improvement: {baseline["I5"]["docsExamined"]} -> {after_i5["docsExamined"]} docs examined ({improvement}% reduction)')
print(f'Note: Average 1.55 events per customer — index reduces scan from 640 to ~1-2 docs')

---
## Cell 9 — Before/After comparison summary

In [ ]:
after = {
    'I1': after_i1,
    'I2': after_i2,
    'I3': after_i3,
    'I4': after_i4,
    'I5': after_i5,
}

queries = {
    'I1': 'find(customer_id)',
    'I2': 'find(home_zone=North)',
    'I3': 'find(maintenance_status=InRepair)',
    'I4': 'find(zone=Airport, status=InRepair)',
    'I5': 'find(customer_id) on op_events',
}

print('=' * 80)
print(f'{"Index":<6} {"Query":<36} {"Before":<10} {"After":<10} {"Improvement"}')
print('-' * 80)

for idx in ['I1', 'I2', 'I3', 'I4', 'I5']:
    b = baseline[idx]['docsExamined']
    a = after[idx]['docsExamined']
    pct = round((1 - a / max(b, 1)) * 100, 1)
    print(f'{idx:<6} {queries[idx]:<36} {b:<10} {a:<10} {pct}% reduction')

print('=' * 80)
print('\nAll queries transitioned from COLLSCAN to IXSCAN after index creation.')

## Cell 10 — List all indexes on each collection

In [ ]:
for coll_name, coll in [('customer_cases', customer_cases),
                         ('fleet_health', fleet_health),
                         ('operational_events', operational_events)]:
    print(f'\n{coll_name} indexes:')
    for idx in coll.list_indexes():
        print(f'  {idx["name"]}: {dict(idx["key"])}')

client.close()
print('\nConnection closed.')